# 01 — EDA: Urinary biomarkers (Debernardi 2020)

Goal: understand the data **before** modeling. Answer every question in a markdown cell below it — these answers go into your README and interview story.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import roc_auc_score
from pancreasignal.data.urine import load_urine, select_task, DIAGNOSIS, TASKS

sns.set_theme(style="whitegrid", context="notebook")
DX_ORDER = ["control", "benign", "pdac"]
PALETTE = {"control": "#4C72B0", "benign": "#DD8452", "pdac": "#C44E52"}

df = load_urine()
df["dx"] = pd.Categorical(df["diagnosis"].map(DIAGNOSIS), categories=DX_ORDER)
print(df.shape)
df.head()

## Q1. How many samples per diagnosis? Per center (`sample_origin`)?
Hint: `pd.crosstab(df.sample_origin, df.dx)`. **Which center has very few PDAC? → that center may be skipped in leave-one-center-out.**

In [ ]:
display(pd.crosstab(df.sample_origin, df.dx, margins=True, margins_name="total"))
display(pd.crosstab(df.patient_cohort, df.dx, margins=True, margins_name="total"))

ct = pd.crosstab(df.sample_origin, df.dx)
ax = ct.plot.barh(stacked=True, color=[PALETTE[c] for c in ct.columns], figsize=(7, 3.2))
ax.set(xlabel="samples", ylabel="center", title="Samples per center and diagnosis")
plt.tight_layout(); plt.show()

In [ ]:
# Which centers can be a leave-one-center-out (LOCO) test fold?
# Rule from evaluation/splits.py: the test center needs both classes and >= 5 PDAC.
# Also check that the *training* side (all other centers) still has negatives to learn from.
rows_ = []
for task in TASKS:
    sub, y = select_task(df, task)
    for center, y_c in y.groupby(sub.sample_origin):
        n_neg, n_pos = int((y_c == 0).sum()), int((y_c == 1).sum())
        rows_.append({
            "task": task, "test center": center, "neg": n_neg, "pdac": n_pos,
            "usable as test fold": n_neg > 0 and n_pos >= 5,
            "negatives left for training": int((y[sub.sample_origin != center] == 0).sum()),
        })
pd.DataFrame(rows_)

**Answer.**
- 590 samples: 183 control, 208 benign, 199 PDAC. Four centers: BPTB 409, LIV 132, ESP 29, UCL 20.
- **All 183 controls come from one center (BPTB).** LIV and ESP are mostly PDAC (70% and 79%), UCL has 0 PDAC. Center is strongly tied to diagnosis, so center-specific quirks (assay batch, sample handling) could leak in as fake signal.
- LOCO for `pdac_vs_benign` / `pdac_vs_rest`: folds BPTB, ESP, LIV. UCL is skipped (0 PDAC). The ESP fold has only 6 negatives, so its AUROC will be very noisy.
- **LOCO cannot work for `pdac_vs_control`:** ESP and LIV have no controls (skipped), and when BPTB is held out the training set has 0 controls. This task needs a different split (e.g. stratified K-fold), and its result must be labelled "within-center".

## Q2. How much is missing? (`plasma_CA19_9`, `REG1A`, `stage`)
Which diagnosis groups are missing CA19-9 most? Does that bias a CA19-9 comparison?

In [ ]:
cols = ["plasma_CA19_9", "REG1A", "stage"]
miss = df[cols].isna().sum().to_frame("n missing")
miss["% missing"] = (miss["n missing"] / len(df) * 100).round(1)
display(miss)

print("% missing by diagnosis")
display((df[cols].isna().groupby(df.dx).mean() * 100).round(1))

print("% CA19-9 missing by center x diagnosis")
display((df.plasma_CA19_9.isna().groupby([df.sample_origin, df.dx]).mean().unstack() * 100).round(0))

has_ca = df.plasma_CA19_9.notna()
print(f"Diagnosis mix (%): all rows vs the {has_ca.sum()} rows that have CA19-9")
pd.DataFrame({
    "all rows": df.dx.value_counts(normalize=True),
    "rows with CA19-9": df.loc[has_ca, "dx"].value_counts(normalize=True),
}).loc[DX_ORDER].mul(100).round(1)

**Answer.**
- CA19-9 is missing for 240/590 rows (41%), REG1A for 284 (48%). `stage` is missing for 391 rows, which is exactly every non-PDAC row: that is "not applicable", not missing data.
- CA19-9 is **not missing at random**: about 50% missing in control and benign vs 25% in PDAC, and 100% missing at ESP and UCL. It was measured more often when cancer was suspected.
- So the 350 rows with CA19-9 are a different population (43% PDAC vs 34% overall, no ESP/UCL). A fair comparison against CA19-9 must use **the same rows** for both the model and CA19-9, never "model on all rows vs CA19-9 on its rows".

## Q3. Distributions of LYVE1, REG1B, TFF1, creatinine by diagnosis
Plot raw vs `np.log1p`. Which looks more normal? (This justifies `log_transform: true`.)

In [ ]:
markers = ["LYVE1", "REG1B", "TFF1", "creatinine"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for j, m in enumerate(markers):
    for i, (label, f) in enumerate([("raw", lambda s: s), ("log1p", np.log1p)]):
        tmp = pd.DataFrame({"dx": df.dx, "value": f(df[m])})
        sns.histplot(data=tmp, x="value", hue="dx", hue_order=DX_ORDER, palette=PALETTE,
                     element="step", stat="density", common_norm=False, bins=30,
                     legend=(i == 0 and j == 0), ax=axes[i, j])
        axes[i, j].set(title=f"{m} ({label})", xlabel="", ylabel="")
plt.tight_layout(); plt.show()

display(pd.DataFrame({"skew raw": df[markers].skew(), "skew log1p": np.log1p(df[markers]).skew()}).round(2))
print("Median by diagnosis")
df.groupby("dx")[markers + ["plasma_CA19_9"]].median().round(2)

**Answer.**
- Raw values are strongly right-skewed (skewness 1.4 to 5.2; a few huge values dominate). After `log1p` the skewness is between -0.8 and 0.6 and REG1B/TFF1 look much closer to normal, which supports `log_transform: true` for a linear model.
- Exception: LYVE1 has a spike at ~0. 111 rows are below 0.01 ng/ml (70 control, 40 benign, 1 PDAC), probably under the assay's detection limit. `log1p` keeps them near 0; plain `log` would turn them into extreme negative outliers (down to -8.9), another reason to use `log1p`.
- PDAC medians are far higher than benign and control: LYVE1 5.6 vs 1.2 vs 0.15, REG1B 123 vs 20 vs 18, TFF1 723 vs 210 vs 60 ng/ml. Benign sits between control and PDAC, which is why `pdac_vs_benign` is the hard task.

## Q4. Does creatinine differ between groups? Does biomarker / creatinine separate the groups better than raw?

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
sns.boxplot(data=df, x="dx", y="creatinine", hue="dx", palette=PALETTE, legend=False, ax=ax)
ax.set(title="Urine creatinine (mg/ml) by diagnosis", xlabel="")
plt.show()
display(df.groupby("dx").creatinine.describe().round(2))

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for j, m in enumerate(["LYVE1", "REG1B", "TFF1"]):
    for i, (label, vals) in enumerate([("log1p(raw)", np.log1p(df[m])),
                                       ("log1p(raw / creatinine)", np.log1p(df[m] / df.creatinine))]):
        tmp = pd.DataFrame({"dx": df.dx, "value": vals})
        sns.boxplot(data=tmp, x="dx", y="value", hue="dx", palette=PALETTE, legend=False, ax=axes[i, j])
        axes[i, j].set(title=f"{m}: {label}", xlabel="", ylabel="")
plt.tight_layout(); plt.show()

In [ ]:
# Univariate AUROC = how well one value alone ranks PDAC above the negatives.
# Nothing is fitted (AUROC only uses the ordering), so this is descriptive EDA,
# NOT a model result. Do not put these numbers on the CV.
def univariate_auroc(task):
    rows, y = select_task(df, task)
    out = {"creatinine": roc_auc_score(y, rows.creatinine)}
    for m in ["LYVE1", "REG1B", "TFF1"]:
        out[f"{m} raw"] = roc_auc_score(y, rows[m])
        out[f"{m} / creatinine"] = roc_auc_score(y, rows[m] / rows.creatinine)
    return pd.Series(out)

pd.DataFrame({task: univariate_auroc(task) for task in TASKS}).round(3)

**Answer.**
- Creatinine is about the same in all groups (median 0.71 to 0.75 mg/ml; AUROC alone is about 0.52, close to chance). That is what we want: it measures how concentrated the urine is, not disease.
- Dividing by creatinine helps LYVE1 and TFF1 (vs control: LYVE1 0.891 → 0.915, TFF1 0.862 → 0.890; vs benign: TFF1 0.722 → 0.741) but slightly hurts REG1B (0.810 → 0.803). On balance it supports `normalize_by_creatinine: true`, but it is worth an ablation in Phase 4 (with vs without).

## Q5. Age and sex by diagnosis
If PDAC patients are much older than controls, the model can 'cheat' using age. How will you check this? (Hint: train an age-only model as another baseline.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
sns.boxplot(data=df, x="dx", y="age", hue="dx", palette=PALETTE, legend=False, ax=axes[0])
axes[0].set(title="Age by diagnosis", xlabel="")
sex = pd.crosstab(df.dx, df.sex, normalize="index").mul(100)
sex.plot.bar(stacked=True, ax=axes[1], color=["#8172B3", "#64B5CD"], rot=0)
axes[1].set(title="Sex by diagnosis (%)", xlabel="", ylabel="%")
plt.tight_layout(); plt.show()

display(df.groupby("dx").age.describe().round(1))
display(sex.round(1))

age_auc = {}
for task in TASKS:
    rows, y = select_task(df, task)
    age_auc[task] = roc_auc_score(y, rows.age)
pd.Series(age_auc, name="AUROC of age alone").round(3)

**Answer.**
- PDAC patients are about 10 years older (mean 66 vs 56 control, 55 benign). Age alone ranks PDAC above the others with AUROC about 0.74 in all three tasks, so a model can look decent by learning "older = cancer".
- Sex is also unbalanced: 58% male in PDAC vs 37% in control.
- Check: in Phase 4, run an **age-only (and age + sex) baseline** with the same LOCO splits. The biomarker model only adds value if it clearly beats that baseline.

## Q6. PDAC stage distribution
Early detection matters most for stage I–II. How many early-stage cases exist? (Later: report sensitivity by stage from the `_oof.csv`.)

In [ ]:
pdac = df[df.dx == "pdac"].copy()
assert pdac.stage.notna().all(), "some PDAC rows have no stage"
STAGE_ORDER = ["I", "IA", "IB", "II", "IIA", "IIB", "III", "IV"]
is_late = pdac.stage.str.startswith(("III", "IV"))
pdac["stage_group"] = np.where(is_late, "III-IV (late)", "I-II (early)")

display(pdac.stage.value_counts().reindex(STAGE_ORDER).to_frame("patients"))
display(pd.crosstab(pdac.sample_origin, pdac.stage_group, margins=True, margins_name="total"))

counts = pdac.stage.value_counts().reindex(STAGE_ORDER)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(STAGE_ORDER, counts.values,
       color=["#C44E52" if s.startswith(("III", "IV")) else "#55A868" for s in STAGE_ORDER])
ax.set(title="PDAC stage (green = I-II early, red = III-IV late)", xlabel="stage", ylabel="patients")
plt.show()

**Answer.**
- All 199 PDAC have a stage. Early (I–II): **102** cases, but most are IIB (68); only **16** are stage I (I/IA/IB). Late (III–IV): 97 (III 76, IV 21).
- ESP records coarse stages ("I", "II") without sub-stage, and LIV has no stage IV. The documentation file lists "IIIB", which does not appear in the data; trust the data.
- With only 16 stage I cases, any "sensitivity for stage I" will have a very wide confidence interval. Report I–II vs III–IV, with the count next to every number.

## Summary (write 5 bullet points)
- **Case-control data, not a screening population.** 590 patients chosen because their diagnosis was already known (183 control, 208 benign, 199 PDAC). PDAC is 34% here vs well under 1% in real screening, so PPV and real-world performance will be much lower than any AUROC suggests.
- **Center is confounded with diagnosis.** All controls are from BPTB, UCL has no PDAC, LIV/ESP are mostly PDAC. LOCO works for `pdac_vs_benign` / `pdac_vs_rest` (folds BPTB, ESP, LIV) but not for `pdac_vs_control`.
- **CA19-9 is missing not at random** (41% overall; 25% in PDAC vs ~50% in others; none at ESP/UCL). Compare against CA19-9 only on the same rows.
- **Age is a shortcut.** PDAC patients are ~10 years older; age alone gives AUROC ~0.74. An age-only baseline is mandatory.
- **Features:** biomarkers are heavily right-skewed (log1p fixes it); creatinine does not differ by group and normalizing by it helps LYVE1/TFF1. Only 16 stage I cases, so early-stage claims must be made carefully.